# Text Feature Engineering Assignment
## Real-world Dataset: Amazon/Flipkart Product Reviews
### Tasks: One Hot Encoding | Bag of Words | TF-IDF | Sentiment Classification

---
## Step 0: Install Dependencies

In [ ]:
# Install required libraries (run once)
# !pip install requests beautifulsoup4 fake-useragent nltk scikit-learn pandas numpy matplotlib seaborn

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import os

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

def scrape_amazon_reviews(product_url: str, max_pages: int = 5) -> list:
    """
    Scrape reviews from an Amazon product page.
    product_url : URL of the Amazon product (review tab).
    max_pages   : Number of review pages to scrape.
    Returns a list of dicts with keys: review_title, review_text, rating.
    """
    all_reviews = []
    for page in range(1, max_pages + 1):
        url = f"{product_url}&pageNumber={page}"
        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)
            soup = BeautifulSoup(resp.text, "html.parser")
            review_divs = soup.find_all("div", {"data-hook": "review"})
            if not review_divs:
                print(f"  No reviews found on page {page}. Stopping.")
                break
            for div in review_divs:
                title_tag = div.find("a", {"data-hook": "review-title"})
                body_tag  = div.find("span", {"data-hook": "review-body"})
                rating_tag = div.find("i", {"data-hook": "review-star-rating"})
                title  = title_tag.get_text(strip=True) if title_tag else ""
                body   = body_tag.get_text(strip=True)  if body_tag  else ""
                rating = rating_tag.get_text(strip=True).split(" ")[0] if rating_tag else "0"
                if body:
                    all_reviews.append({
                        "review_title": title,
                        "review_text" : body,
                        "rating"      : float(rating)
                    })
            print(f"  Scraped page {page}: {len(review_divs)} reviews")
            time.sleep(random.uniform(1.5, 3.0))   # polite delay
        except Exception as e:
            print(f"  Error on page {page}: {e}")
            break
    return all_reviews

# ── EXAMPLE PRODUCT URL ────────────────────────────────────────────────────────
# 
# The URL below points to a Apple iPhone 15 (128 GB) - Black  reviews.
PRODUCT_REVIEW_URL = (
    "https://www.amazon.in/product-reviews/B0CHX1W1XY"
)

CSV_PATH = "reviews_dataset.csv"
print("Dataset CSV already exists — loading from disk.")
df_raw = pd.read_csv(CSV_PATH);

# Uncomment the block below to scrape reviews if the CSV doesn't exist.
# if os.path.exists(CSV_PATH):
#     print("Dataset CSV already exists — loading from disk.")
#     df_raw = pd.read_csv(CSV_PATH)
# else:
#     print("Scraping Amazon reviews …")
#     reviews = scrape_amazon_reviews(PRODUCT_REVIEW_URL, max_pages=10)
#     df_raw  = pd.DataFrame(reviews)
#     print(f"\nTotal reviews scraped: {len(df_raw)}")

print(df_raw.head(3))

Dataset CSV already exists — loading from disk.
                                         review_text  rating
0  The battery life is absolutely amazing on this...       5
1  Camera quality is brilliant. Night mode photos...       5
2  Very fast processor. Games run smoothly withou...       5


In [10]:
# ── FALLBACK: Use synthetic realistic reviews if scraping fails ────────────────
# This block runs only when scraping returns fewer than 10 reviews.
FALLBACK_REVIEWS = [
    ("The battery life is absolutely amazing on this phone. Lasts two full days easily.", 5),
    ("Camera quality is brilliant. Night mode photos look stunning compared to my old phone.", 5),
    ("Very fast processor. Games run smoothly without any lag or heating issues.", 5),
    ("Build quality feels premium. The glass back looks elegant and feels solid.", 4),
    ("Display is vibrant and crisp. Watching movies on this screen is a pleasure.", 5),
    ("Charging is super fast. Goes from zero to hundred in about forty five minutes.", 4),
    ("Software is clean and bloatware free. Updates are regular and timely.", 4),
    ("The speaker sound is loud and clear. Bass is decent for a phone speaker.", 4),
    ("Delivery was fast and packaging was excellent. Product was well protected.", 5),
    ("Great value for money. Best phone you can buy at this price range.", 5),
    ("Battery drains quickly when using mobile data and GPS simultaneously.", 2),
    ("Camera has too much sharpening. Photos look over processed and artificial.", 2),
    ("Phone gets warm during gaming sessions. Thermal management could be better.", 3),
    ("No headphone jack is a major disappointment. Should have included one.", 2),
    ("Fingerprint sensor is slow and sometimes fails to recognize my finger.", 2),
    ("The phone slips easily due to the smooth glass back. A case is necessary.", 3),
    ("Received a defective unit. Screen had dead pixels. Had to return it.", 1),
    ("Customer service was unhelpful when I reported the issue. Very frustrating.", 1),
    ("After two months the charging port became loose. Poor quality control.", 1),
    ("Wi-Fi signal drops frequently in certain areas. Software bug not yet fixed.", 2),
    ("Average phone for the price. Nothing stands out as truly impressive.", 3),
    ("Photos are good during daylight but night shots are blurry and noisy.", 3),
    ("Screen brightness is low outdoors. Hard to see under direct sunlight.", 3),
    ("The UI is okay but occasional stutters ruin the smooth experience.", 3),
    ("Decent phone overall but the competition offers more at the same price.", 3),
    ("Excellent screen to body ratio. Bezels are thin and the display is gorgeous.", 5),
    ("Performance is top notch. Multitasking between heavy apps is seamless.", 5),
    ("Sound quality during calls is crystal clear. No background noise issues.", 4),
    ("Face unlock works perfectly even in low light conditions. Very convenient.", 4),
    ("Gyroscope sensor is accurate. Gaming experience is immersive and fun.", 5),
    ("5G connectivity is blazing fast. Streaming 4K videos without any buffering.", 5),
    ("Dual SIM support is great for managing personal and work numbers separately.", 4),
    ("Bluetooth connectivity is stable. Paired with earbuds without any drops.", 4),
    ("Storage options are flexible. 256 GB variant is perfect for heavy users.", 4),
    ("Color options are beautiful. The midnight black variant looks premium.", 4),
    ("Battery backup has reduced significantly after four months of use.", 2),
    ("The phone lags after the latest software update. Needs a fix urgently.", 2),
    ("Macro camera is useless. Photos come out blurry even in good lighting.", 2),
    ("Volume buttons feel flimsy and cheap compared to the rest of the build.", 2),
    ("App drawer organization is confusing and not intuitive to use daily.", 2),
    ("Received the wrong color. Ordered black but got blue. Disappointing error.", 1),
    ("The seller delivered a used phone instead of a brand new sealed unit.", 1),
    ("Screen protection glass cracked on first minor drop. Very fragile design.", 1),
    ("Overpriced for what it offers. Better phones available at lower prices.", 1),
    ("Do not buy. The phone stopped working after three weeks of light use.", 1),
    ("This phone exceeded all my expectations. Truly a flagship killer product.", 5),
    ("Highly recommend this to anyone looking for a reliable daily driver phone.", 5),
    ("Worth every rupee spent. My family members also want to buy the same.", 5),
    ("Outstanding performance and camera combo. Nothing beats it at this price.", 5),
    ("Switching from iPhone to this was the best decision I made this year.", 5),
    ("Good phone but lacks wireless charging which competitors offer at same price.", 3),
    ("Build is solid but the notch design feels outdated in this era of phones.", 3),
    ("Call quality is clear but earpiece volume is low during outdoor calls.", 3),
    ("Battery is fine for moderate users but heavy users need to charge daily.", 3),
    ("Performance is good for regular tasks but stutters in intensive workloads.", 3),
    ("Super impressed with the camera zoom capabilities. Portraits look amazing.", 5),
    ("The ultra wide camera captures stunning landscape shots with great detail.", 5),
    ("Video recording is extremely smooth thanks to optical image stabilisation.", 5),
    ("Dolby Atmos speakers deliver an incredible audio experience for movies.", 5),
    ("Gorilla Glass protection gives confidence. Survived a two feet drop fine.", 4),
    ("IP68 water resistance is reassuring. Survived rain splashes with no issues.", 4),
    ("NFC payments work flawlessly. Tap and pay at all major retail stores.", 4),
    ("The always on display feature is handy for checking time and notifications.", 4),
    ("Haptic feedback is satisfying and premium. Typing feels responsive and nice.", 4),
    ("GPS locks on quickly. Navigation during road trips was accurate throughout.", 4),
    ("Terrible after sales service. Service center kept the phone for one month.", 1),
    ("Phone restarted randomly multiple times daily. Major inconvenience always.", 1),
    ("The promised software update never arrived. Brand does not keep promises.", 1),
    ("Microphone picks up a lot of wind noise during outdoor voice recordings.", 2),
    ("Earpiece has slight distortion at maximum volume. Annoying during calls.", 2),
    ("Heating during charging is excessive. Feels very hot to hold while charging.", 2),
    ("App crashes frequently. Cleared cache but the problem keeps coming back.", 2),
    ("Proximity sensor malfunction causes screen to turn on in pocket randomly.", 2),
    ("The notification LED is missing. Hard to know if I have missed messages.", 3),
    ("Screen refresh rate drops in power saving mode more than I would like.", 3),
    ("Dark mode implementation is good but some system apps still look bright.", 3),
    ("Comes with a charger in the box which many premium brands no longer do.", 4),
    ("Setup process was quick and easy. Data transfer from old phone was smooth.", 4),
    ("Gaming mode is a great feature. It blocks notifications during gaming sessions.", 4),
    ("Split screen multitasking works well. Useful for productivity on the go.", 4),
    ("Auto brightness adjustment is smart and works accurately in all conditions.", 4),
    ("The bokeh effect in portrait mode is natural and not overdone at all.", 5),
    ("Wide aperture mode creates beautiful depth of field in close up photos.", 5),
    ("Pro mode gives great manual control for photography enthusiasts like me.", 5),
    ("Slow motion video at 960 frames per second is absolutely breathtaking.", 5),
    ("Astrophotography mode captures stars surprisingly well for a phone camera.", 5),
    ("Vlogging with this phone is a breeze. Front camera quality is outstanding.", 5),
    ("YouTube consumption on this display is pure bliss. Colours are accurate.", 5),
    ("Reading ebooks on the screen is comfortable. Eye care mode helps a lot.", 4),
    ("Kids love using this phone for educational apps. Performance never lags.", 4),
    ("The parental control features are robust and easy to configure for kids.", 4),
    ("Calling quality over VoLTE is excellent. Crystal clear even in weak signals.", 4),
    ("Roaming data works well internationally. Used it in Dubai without issues.", 4),
    ("The phone maintained peak performance even after six months of use easily.", 5),
    ("Regular security patches keep the phone safe. Good brand commitment seen.", 5),
    ("Ecosystem integration with earbuds and smartwatch is seamless and intuitive.", 5),
    ("Gaming tournament was won because of this phone's low latency touch response.", 5),
    ("Recommended by my tech youtuber. Every claim they made was absolutely true.", 5),
]

if not os.path.exists(CSV_PATH) or len(df_raw) < 10:
    print("Using fallback synthetic dataset …")
    df_raw = pd.DataFrame(FALLBACK_REVIEWS, columns=["review_text", "rating"])

# Ensure required column exists
assert "review_text" in df_raw.columns, "CSV must contain a 'review_text' column!"
df_raw = df_raw.dropna(subset=["review_text"]).reset_index(drop=True)
df_raw.to_csv(CSV_PATH, index=False)

print(f"\nDataset shape : {df_raw.shape}")
print(f"Columns       : {list(df_raw.columns)}")
df_raw.head(5)

PermissionError: [Errno 13] Permission denied: 'reviews_dataset.csv'

In [ ]:
# This block runs only when scraping returns fewer than 10 reviews.
FALLBACK_REVIEWS = [
    ("The battery life is absolutely amazing on this phone. Lasts two full days easily.", 5),
    ("Camera quality is brilliant. Night mode photos look stunning compared to my old phone.", 5),
    ("Very fast processor. Games run smoothly without any lag or heating issues.", 5),
    ("Build quality feels premium. The glass back looks elegant and feels solid.", 4),
    ("Display is vibrant and crisp. Watching movies on this screen is a pleasure.", 5),
    ("Charging is super fast. Goes from zero to hundred in about forty five minutes.", 4),
    ("Software is clean and bloatware free. Updates are regular and timely.", 4),
    ("The speaker sound is loud and clear. Bass is decent for a phone speaker.", 4),
    ("Delivery was fast and packaging was excellent. Product was well protected.", 5),
    ("Great value for money. Best phone you can buy at this price range.", 5),
    ("Battery drains quickly when using mobile data and GPS simultaneously.", 2),
    ("Camera has too much sharpening. Photos look over processed and artificial.", 2),
    ("Phone gets warm during gaming sessions. Thermal management could be better.", 3),
    ("No headphone jack is a major disappointment. Should have included one.", 2),
    ("Fingerprint sensor is slow and sometimes fails to recognize my finger.", 2),
    ("The phone slips easily due to the smooth glass back. A case is necessary.", 3),
    ("Received a defective unit. Screen had dead pixels. Had to return it.", 1),
    ("Customer service was unhelpful when I reported the issue. Very frustrating.", 1),
    ("After two months the charging port became loose. Poor quality control.", 1),
    ("Wi-Fi signal drops frequently in certain areas. Software bug not yet fixed.", 2),
    ("Average phone for the price. Nothing stands out as truly impressive.", 3),
    ("Photos are good during daylight but night shots are blurry and noisy.", 3),
    ("Screen brightness is low outdoors. Hard to see under direct sunlight.", 3),
    ("The UI is okay but occasional stutters ruin the smooth experience.", 3),
    ("Decent phone overall but the competition offers more at the same price.", 3),
    ("Excellent screen to body ratio. Bezels are thin and the display is gorgeous.", 5),
    ("Performance is top notch. Multitasking between heavy apps is seamless.", 5),
    ("Sound quality during calls is crystal clear. No background noise issues.", 4),
    ("Face unlock works perfectly even in low light conditions. Very convenient.", 4),
    ("Gyroscope sensor is accurate. Gaming experience is immersive and fun.", 5),
    ("5G connectivity is blazing fast. Streaming 4K videos without any buffering.", 5),
    ("Dual SIM support is great for managing personal and work numbers separately.", 4),
    ("Bluetooth connectivity is stable. Paired with earbuds without any drops.", 4),
    ("Storage options are flexible. 256 GB variant is perfect for heavy users.", 4),
    ("Color options are beautiful. The midnight black variant looks premium.", 4),
    ("Battery backup has reduced significantly after four months of use.", 2),
    ("The phone lags after the latest software update. Needs a fix urgently.", 2),
    ("Macro camera is useless. Photos come out blurry even in good lighting.", 2),
    ("Volume buttons feel flimsy and cheap compared to the rest of the build.", 2),
    ("App drawer organization is confusing and not intuitive to use daily.", 2),
    ("Received the wrong color. Ordered black but got blue. Disappointing error.", 1),
    ("The seller delivered a used phone instead of a brand new sealed unit.", 1),
    ("Screen protection glass cracked on first minor drop. Very fragile design.", 1),
    ("Overpriced for what it offers. Better phones available at lower prices.", 1),
    ("Do not buy. The phone stopped working after three weeks of light use.", 1),
    ("This phone exceeded all my expectations. Truly a flagship killer product.", 5),
    ("Highly recommend this to anyone looking for a reliable daily driver phone.", 5),
    ("Worth every rupee spent. My family members also want to buy the same.", 5),
    ("Outstanding performance and camera combo. Nothing beats it at this price.", 5),
    ("Switching from iPhone to this was the best decision I made this year.", 5),
    ("Good phone but lacks wireless charging which competitors offer at same price.", 3),
    ("Build is solid but the notch design feels outdated in this era of phones.", 3),
    ("Call quality is clear but earpiece volume is low during outdoor calls.", 3),
    ("Battery is fine for moderate users but heavy users need to charge daily.", 3),
    ("Performance is good for regular tasks but stutters in intensive workloads.", 3),
    ("Super impressed with the camera zoom capabilities. Portraits look amazing.", 5),
    ("The ultra wide camera captures stunning landscape shots with great detail.", 5),
    ("Video recording is extremely smooth thanks to optical image stabilisation.", 5),
    ("Dolby Atmos speakers deliver an incredible audio experience for movies.", 5),
    ("Gorilla Glass protection gives confidence. Survived a two feet drop fine.", 4),
    ("IP68 water resistance is reassuring. Survived rain splashes with no issues.", 4),
    ("NFC payments work flawlessly. Tap and pay at all major retail stores.", 4),
    ("The always on display feature is handy for checking time and notifications.", 4),
    ("Haptic feedback is satisfying and premium. Typing feels responsive and nice.", 4),
    ("GPS locks on quickly. Navigation during road trips was accurate throughout.", 4),
    ("Terrible after sales service. Service center kept the phone for one month.", 1),
    ("Phone restarted randomly multiple times daily. Major inconvenience always.", 1),
    ("The promised software update never arrived. Brand does not keep promises.", 1),
    ("Microphone picks up a lot of wind noise during outdoor voice recordings.", 2),
    ("Earpiece has slight distortion at maximum volume. Annoying during calls.", 2),
    ("Heating during charging is excessive. Feels very hot to hold while charging.", 2),
    ("App crashes frequently. Cleared cache but the problem keeps coming back.", 2),
    ("Proximity sensor malfunction causes screen to turn on in pocket randomly.", 2),
    ("The notification LED is missing. Hard to know if I have missed messages.", 3),
    ("Screen refresh rate drops in power saving mode more than I would like.", 3),
    ("Dark mode implementation is good but some system apps still look bright.", 3),
    ("Comes with a charger in the box which many premium brands no longer do.", 4),
    ("Setup process was quick and easy. Data transfer from old phone was smooth.", 4),
    ("Gaming mode is a great feature. It blocks notifications during gaming sessions.", 4),
    ("Split screen multitasking works well. Useful for productivity on the go.", 4),
    ("Auto brightness adjustment is smart and works accurately in all conditions.", 4),
    ("The bokeh effect in portrait mode is natural and not overdone at all.", 5),
    ("Wide aperture mode creates beautiful depth of field in close up photos.", 5),
    ("Pro mode gives great manual control for photography enthusiasts like me.", 5),
    ("Slow motion video at 960 frames per second is absolutely breathtaking.", 5),
    ("Astrophotography mode captures stars surprisingly well for a phone camera.", 5),
    ("Vlogging with this phone is a breeze. Front camera quality is outstanding.", 5),
    ("YouTube consumption on this display is pure bliss. Colours are accurate.", 5),
    ("Reading ebooks on the screen is comfortable. Eye care mode helps a lot.", 4),
    ("Kids love using this phone for educational apps. Performance never lags.", 4),
    ("The parental control features are robust and easy to configure for kids.", 4),
    ("Calling quality over VoLTE is excellent. Crystal clear even in weak signals.", 4),
    ("Roaming data works well internationally. Used it in Dubai without issues.", 4),
    ("The phone maintained peak performance even after six months of use easily.", 5),
    ("Regular security patches keep the phone safe. Good brand commitment seen.", 5),
    ("Ecosystem integration with earbuds and smartwatch is seamless and intuitive.", 5),
    ("Gaming tournament was won because of this phone's low latency touch response.", 5),
    ("Recommended by my tech youtuber. Every claim they made was absolutely true.", 5),
]

if not os.path.exists(CSV_PATH) or len(df_raw) < 10:
    print("Using fallback synthetic dataset …")
    df_raw = pd.DataFrame(FALLBACK_REVIEWS, columns=["review_text", "rating"])

# Ensure required column exists
assert "review_text" in df_raw.columns, "CSV must contain a 'review_text' column!"
df_raw = df_raw.dropna(subset=["review_text"]).reset_index(drop=True)
df_raw.to_csv(CSV_PATH, index=False)

print(f"\nDataset shape : {df_raw.shape}")
print(f"Columns       : {list(df_raw.columns)}")
df_raw.head(5)

In [5]:
df_raw.info()


<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   review_text  100 non-null    str  
 1   rating       100 non-null    int64
dtypes: int64(1), str(1)
memory usage: 8.8 KB


In [8]:
df_raw.duplicated().sum()

0

In [7]:
df_raw=df_raw.drop_duplicates().reset_index(drop=True)
print(f"Dataset shape after removing duplicates: {df_raw.shape}")

Dataset shape after removing duplicates: (98, 2)


---
## Task 1: Text Preprocessing

In [46]:
import re
import string

stop_words = {"and", "a", "the"}
def preprocess(text: str)->list[str]:
             
    """
    Full preprocessing pipeline:
      1. Lowercase
      2. Remove punctuation & special characters
      3. Tokenize   
    """
    # 1. Lowercase
    text = text.lower()
    # 2. Remove punctuation & digits
    text = re.sub(r"[^a-z\s]", "", text)
    # 3. Tokenize
    tokens = text.split()
    # 4. Remove stopwords (optional)
    return [tok for tok in tokens if tok not in stop_words]
   
  
df_raw["processed_text"] = df_raw["review_text"].apply(preprocess)


all_tokens = [tok for tokens in df_raw["processed_text"] for tok in tokens]
vocabulary  = sorted(set(all_tokens))          # unique words, alphabetically sorted
vocab_index = {word: idx for idx, word in enumerate(vocabulary)}  # word → position
print(f"Vocabulary size: {len(vocabulary)}")
print(f"Sample words  : {vocabulary[:583]}")


# Preview
# print("=== BEFORE vs AFTER PREPROCESSING ===")

# for i in range(98):
#     print(f"\n[{i+1}] ORIGINAL : {df_raw['review_text'].iloc[i]}")
#     print(f"    PROCESSED: {df_raw['processed_text'].iloc[i]}")

Vocabulary size: 583
Sample words  : ['about', 'absolutely', 'accurate', 'accurately', 'adjustment', 'after', 'all', 'also', 'always', 'amazing', 'an', 'annoying', 'any', 'anyone', 'aperture', 'app', 'apps', 'are', 'areas', 'arrived', 'artificial', 'as', 'astrophotography', 'at', 'atmos', 'audio', 'auto', 'available', 'average', 'back', 'background', 'backup', 'bass', 'battery', 'be', 'beats', 'beautiful', 'became', 'because', 'best', 'better', 'between', 'bezels', 'black', 'blazing', 'bliss', 'bloatware', 'blocks', 'blue', 'bluetooth', 'blurry', 'body', 'bokeh', 'box', 'brand', 'brands', 'breathtaking', 'breeze', 'bright', 'brightness', 'brilliant', 'buffering', 'bug', 'build', 'but', 'buttons', 'buy', 'by', 'cache', 'call', 'calling', 'calls', 'camera', 'can', 'capabilities', 'captures', 'care', 'case', 'causes', 'center', 'certain', 'charge', 'charger', 'charging', 'cheap', 'checking', 'claim', 'clean', 'clear', 'cleared', 'close', 'color', 'colours', 'combo', 'come', 'comes', 'comf

---
## Task 2: Vocabulary Creation

In [35]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(
    tokenizer=lambda x: x,
    preprocessor=lambda x: x,
    lowercase=False
)
cv.fit(df_raw["processed_text"])

vocab = cv.get_feature_names_out()
print("Vocabulary size:", len(vocab))
print(vocab[:50])

Vocabulary size: 583
['about' 'absolutely' 'accurate' 'accurately' 'adjustment' 'after' 'all'
 'also' 'always' 'amazing' 'an' 'annoying' 'any' 'anyone' 'aperture' 'app'
 'apps' 'are' 'areas' 'arrived' 'artificial' 'as' 'astrophotography' 'at'
 'atmos' 'audio' 'auto' 'available' 'average' 'back' 'background' 'backup'
 'bass' 'battery' 'be' 'beats' 'beautiful' 'became' 'because' 'best'
 'better' 'between' 'bezels' 'black' 'blazing' 'bliss' 'bloatware'
 'blocks' 'blue' 'bluetooth']


c:\Mine\AI\Full Stack Gen AI  BootCamp (KrishNaik)\Practicals\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


---
## Task 3: Feature Engineering
### 3a. One-Hot Encoding (Document-Level Vector)

In [42]:
import numpy as np
from sklearn.preprocessing import OneHotEncoder

from sklearn.preprocessing import MultiLabelBinarizer

encoder=OneHotEncoder(sparse_output=False)
# df_tokens = df_raw.explode("processed_text")[["processed_text"]]
# X_tokens = encoder.fit_transform([[wor] for wor in df_tokens])
# print("Encoded feature shape:", X_tokens.shape)

df_tokens = df_raw.explode("processed_text")[["processed_text"]]
X_tokens = encoder.fit_transform(df_tokens)
print("Encoded feature shape:", X_tokens.shape)
print("Encoded :", X_tokens)


Encoded feature shape: (1057, 583)
Encoded : [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


### 3b. Bag of Words using CountVectorizer

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=500)   # limit vocab to top 500 for efficiency
bow_matrix = cv.fit_transform(df_raw["processed_text"])

print(f"Bag of Words Matrix Shape   : {bow_matrix.shape}")
print(f"Matrix format               : {type(bow_matrix)}")
print(f"Vocabulary size (top 500)   : {len(cv.get_feature_names_out())}")
print(f"\nSample BoW feature names (first 20):")
print(list(cv.get_feature_names_out()[:20]))

# Show BoW vector for first document
import pandas as pd
bow_df = pd.DataFrame(
    bow_matrix[0].toarray(),
    columns=cv.get_feature_names_out()
)
non_zero = bow_df.loc[:, (bow_df != 0).any(axis=0)]
print(f"\nBoW for document 1 (non-zero only):")
print(non_zero.to_string())

### 3c. TF-IDF using TfidfVectorizer

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=500)
tfidf_matrix = tfidf.fit_transform(df_raw["processed_text"])

print(f"TF-IDF Matrix Shape : {tfidf_matrix.shape}")
print(f"Matrix format       : {type(tfidf_matrix)}")

# Top TF-IDF words in first document
feature_names = tfidf.get_feature_names_out()
first_doc_scores = tfidf_matrix[0].toarray().flatten()
top_indices = first_doc_scores.argsort()[::-1][:10]

print("\nTop 10 TF-IDF words in Document 1:")
for idx in top_indices:
    if first_doc_scores[idx] > 0:
        print(f"  {feature_names[idx]:<20} {first_doc_scores[idx]:.4f}")

---
## Task 4: Comparison Analysis

In [ ]:
comparison_data = {
    "Feature"        : ["Method", "Vector values", "Captures frequency?",
                        "Penalises common words?", "Vector size",
                        "Handles unseen words?", "Best for"],
    "One-Hot Encoding": ["OHE", "Binary (0/1)", "No",
                         "No", "vocab_size per doc",
                         "No (UNK ignored)", "Small vocab, simple classification"],
    "Bag of Words"   : ["BoW", "Integer counts", "Yes",
                         "No", "vocab_size per doc",
                         "No", "Frequency-based tasks, Naive Bayes"],
    "TF-IDF"         : ["TF-IDF", "Float (0-1 normalised)", "Yes (TF)",
                         "Yes (IDF down-weights common)", "vocab_size per doc",
                         "No", "Information retrieval, importance ranking"],
}

comparison_df = pd.DataFrame(comparison_data).set_index("Feature")
print("=== OHE vs BoW vs TF-IDF Comparison ===")
print(comparison_df.to_string())

# ── Why common words get low TF-IDF weight ────────────────────────────────────
print("""
─────────────────────────────────────────────────────────────────────────────
WHY COMMON WORDS RECEIVE LOWER TF-IDF WEIGHT
─────────────────────────────────────────────────────────────────────────────
TF-IDF = TF(t,d) × IDF(t)

  TF  (Term Frequency)       = count(t in d) / total_tokens_in_d
  IDF (Inverse Doc Frequency) = log(N / df(t))   where N = total docs

Words like 'phone', 'good', 'nice' appear in ALMOST EVERY document.
  → df(t) ≈ N  →  IDF ≈ log(1) = 0  →  TF-IDF ≈ 0

Rare but meaningful words like 'astrophotography', 'gyroscope' appear
in only a few documents:
  → df(t) is small  →  IDF is large  →  TF-IDF is high

This makes TF-IDF superior to BoW for identifying the *important* words
that actually distinguish one document from another.
─────────────────────────────────────────────────────────────────────────────
""")

In [ ]:
# Visualise TF-IDF scores for top words across entire corpus
mean_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).flatten()
top_idx    = mean_tfidf.argsort()[::-1][:20]
top_words  = [feature_names[i] for i in top_idx]
top_scores = mean_tfidf[top_idx]

plt.figure(figsize=(13, 5))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(top_words)))
bars   = plt.barh(top_words[::-1], top_scores[::-1], color=colors[::-1],
                  edgecolor="black", height=0.7)
plt.title("Top 20 Words by Mean TF-IDF Score (Corpus-wide)",
          fontsize=14, fontweight="bold")
plt.xlabel("Mean TF-IDF Score")
plt.tight_layout()
plt.savefig("tfidf_top20.png", dpi=150)
plt.show()
print("Plot saved as tfidf_top20.png")

---
## Task 5: Sparse Matrix Analysis

In [ ]:
def sparsity(matrix) -> float:
    """Percentage of zero elements."""
    total = matrix.shape[0] * matrix.shape[1]
    # For scipy sparse matrices
    try:
        nonzero = matrix.nnz
    except AttributeError:
        nonzero = np.count_nonzero(matrix)
    return (1 - nonzero / total) * 100

print("=" * 60)
print("SPARSE MATRIX ANALYSIS")
print("=" * 60)

print(f"\n{'Matrix':<20} {'Shape':<20} {'Non-zeros':<15} {'Sparsity %'}")
print("-" * 60)

# OHE (dense numpy array)
ohe_nz  = np.count_nonzero(ohe_matrix)
ohe_sp  = (1 - ohe_nz / ohe_matrix.size) * 100
print(f"{'OHE':<20} {str(ohe_matrix.shape):<20} {ohe_nz:<15} {ohe_sp:.2f}%")

# BoW (sparse)
print(f"{'Bag of Words':<20} {str(bow_matrix.shape):<20} {bow_matrix.nnz:<15} {sparsity(bow_matrix):.2f}%")

# TF-IDF (sparse)
print(f"{'TF-IDF':<20} {str(tfidf_matrix.shape):<20} {tfidf_matrix.nnz:<15} {sparsity(tfidf_matrix):.2f}%")

print("""
─────────────────────────────────────────────────────────────────────────────
WHY SPARSE MATRICES ARE INEFFICIENT FOR LARGE-SCALE SYSTEMS
─────────────────────────────────────────────────────────────────────────────
1. MEMORY: A dense matrix for 1M docs × 100K vocab = 100 BILLION floats
   (≈ 400 GB). 95%+ of values are zeros — pure waste.

2. COMPUTATION: Matrix multiplications iterate over millions of useless
   zero values, drastically increasing CPU/GPU time.

3. BANDWIDTH: Transferring dense matrices across network (distributed
   systems) is prohibitively slow.

SOLUTIONS in industry:
  • CSR/CSC sparse formats (scipy.sparse) store only non-zero values.
  • Dense Word Embeddings (Word2Vec, FastText, BERT) map words into
    compact 100-300 dimensional DENSE vectors, eliminating sparsity.
─────────────────────────────────────────────────────────────────────────────
""")

---
## Task 6: Real-world Questions

In [ ]:
print("""
══════════════════════════════════════════════════════════════════════════════
Q1: WHY BAG OF WORDS FAILS TO UNDERSTAND SEMANTIC MEANING
══════════════════════════════════════════════════════════════════════════════
BoW treats each word as an independent atomic unit with no relationship
to any other word. It has NO concept of meaning.

Example:
  "The phone is amazing"      → BoW sees: {phone:1, amazing:1}
  "The smartphone is awesome" → BoW sees: {smartphone:1, awesome:1}

These two sentences are SEMANTICALLY IDENTICAL but BoW gives them
ZERO overlap (cosine similarity = 0) because they share no words.

Other failures:
  • Synonyms  : 'buy' ≠ 'purchase' in BoW space
  • Antonyms  : 'great' vs 'not great' — BoW ignores negation context
  • Word order: 'dog bites man' = 'man bites dog' for BoW (WRONG!)
  • Polysemy  : 'bank' (river) = 'bank' (finance) — same vector

Solution: Word embeddings (Word2Vec, GloVe, BERT) encode semantic
meaning into dense continuous vectors.

══════════════════════════════════════════════════════════════════════════════
Q2: WHEN TO USE BOW vs TF-IDF IN INDUSTRY
══════════════════════════════════════════════════════════════════════════════
Use Bag of Words when:
  • You need simplicity and fast training (e.g. spam detection)
  • Document classification with Naive Bayes
  • Small datasets where frequency counts are sufficient
  • When all words are roughly equally important for the task

Use TF-IDF when:
  • Information retrieval / search engines (ranking relevance)
  • Keyword extraction from documents
  • Document clustering and topic modelling
  • When common words ('the', 'a', 'is') would distort results in BoW
  • Recommendation systems based on content similarity

══════════════════════════════════════════════════════════════════════════════
Q3: LIMITATIONS OF TF-IDF IN REAL APPLICATIONS
══════════════════════════════════════════════════════════════════════════════
1. No semantic understanding: 'happy' ≠ 'joyful' — same problem as BoW
2. Fixed vocabulary: Cannot handle new words (OOV problem)
3. Document length bias: Long documents naturally have higher term
   frequencies even for irrelevant terms
4. No word order / syntax: Loses grammatical structure and context
5. Corpus-dependent: IDF scores change when the corpus changes;
   model must be retrained when new documents are added
6. High dimensionality: Vocabulary of 100K+ words creates huge sparse
   vectors, expensive for storage and computation
7. Multi-word expressions: 'New York' is treated as 'New' + 'York'
   separately — loses the phrase meaning
══════════════════════════════════════════════════════════════════════════════
""")

---
## Task 7: Mini Use Case — Sentiment Classification

In [ ]:
from sklearn.model_selection      import train_test_split, cross_val_score
from sklearn.linear_model         import LogisticRegression
from sklearn.naive_bayes          import MultinomialNB
from sklearn.metrics              import (
    accuracy_score, classification_report, confusion_matrix
)
import seaborn as sns

# ── Create sentiment labels ────────────────────────────────────────────────────
# Ratings 4-5 = Positive (1), Ratings 1-2 = Negative (0), 3 = excluded
if "rating" in df_raw.columns:
    df_model = df_raw[df_raw["rating"] != 3].copy()
    df_model["sentiment"] = (df_model["rating"] >= 4).astype(int)
else:
    # If no rating column, assign first half as positive, second half negative
    df_model = df_raw.copy()
    df_model["sentiment"] = [1 if i < len(df_model)//2 else 0
                              for i in range(len(df_model))]

print(f"Dataset for classification:")
print(f"  Total samples : {len(df_model)}")
print(f"  Positive (1)  : {df_model['sentiment'].sum()}")
print(f"  Negative (0)  : {(df_model['sentiment'] == 0).sum()}")

X_text = df_model["processed_text"]
y      = df_model["sentiment"]

In [ ]:
# ── Feature extraction ────────────────────────────────────────────────────────
cv_clf    = CountVectorizer(max_features=500)
tfidf_clf = TfidfVectorizer(max_features=500)

X_bow   = cv_clf.fit_transform(X_text)
X_tfidf = tfidf_clf.fit_transform(X_text)

# Train/Test split
X_bow_tr,   X_bow_te,   y_tr, y_te = train_test_split(
    X_bow,   y, test_size=0.25, random_state=42, stratify=y)
X_tfidf_tr, X_tfidf_te, _,    _    = train_test_split(
    X_tfidf, y, test_size=0.25, random_state=42, stratify=y)

print(f"Train size: {X_bow_tr.shape[0]}  |  Test size: {X_bow_te.shape[0]}")

In [ ]:
# ── Model training & evaluation ───────────────────────────────────────────────
results = []

configs = [
    ("Logistic Regression", "BoW",   LogisticRegression(max_iter=1000), X_bow_tr,   X_bow_te),
    ("Logistic Regression", "TF-IDF",LogisticRegression(max_iter=1000), X_tfidf_tr, X_tfidf_te),
    ("Naive Bayes",         "BoW",   MultinomialNB(),                   X_bow_tr,   X_bow_te),
    ("Naive Bayes",         "TF-IDF",MultinomialNB(),                   X_tfidf_tr, X_tfidf_te),
]

for model_name, features, clf, X_tr, X_te in configs:
    clf.fit(X_tr, y_tr)
    y_pred    = clf.predict(X_te)
    acc       = accuracy_score(y_te, y_pred)
    cv_scores = cross_val_score(clf, X_tr, y_tr, cv=5, scoring="accuracy")
    results.append({
        "Model"          : model_name,
        "Features"       : features,
        "Test Accuracy"  : f"{acc:.4f}",
        "CV Mean±Std"    : f"{cv_scores.mean():.4f} ± {cv_scores.std():.4f}",
    })
    print(f"\n{'='*55}")
    print(f"{model_name} + {features}")
    print(f"{'='*55}")
    print(f"Test Accuracy  : {acc:.4f}")
    print(f"CV  Accuracy   : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(classification_report(y_te, y_pred,
          target_names=["Negative","Positive"]))

results_df = pd.DataFrame(results)
print("\n=== SUMMARY TABLE ===")
print(results_df.to_string(index=False))

In [ ]:
# ── Confusion Matrix Heatmaps ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (model_name, features, clf, X_tr, X_te) in zip(axes, configs):
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    cm     = confusion_matrix(y_te, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Neg","Pos"],
                yticklabels=["Neg","Pos"])
    ax.set_title(f"{model_name}\n({features})  Acc={accuracy_score(y_te,y_pred):.3f}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.suptitle("Confusion Matrices — Sentiment Classification",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved as confusion_matrices.png")

In [ ]:
# ── Accuracy Comparison Bar Chart ─────────────────────────────────────────────
labels  = [f"{r['Model']}\n({r['Features']})" for _, r in results_df.iterrows()]
accs    = [float(r["Test Accuracy"])           for _, r in results_df.iterrows()]
colors  = ["#2196F3", "#FF5722", "#4CAF50", "#9C27B0"]

plt.figure(figsize=(10, 5))
bars = plt.bar(labels, accs, color=colors, edgecolor="black", width=0.5)
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.005,
             f"{acc:.4f}", ha="center", va="bottom", fontsize=11,
             fontweight="bold")
plt.ylim(0, 1.1)
plt.ylabel("Test Accuracy", fontsize=12)
plt.title("Model Accuracy: BoW vs TF-IDF Features",
          fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("accuracy_comparison.png", dpi=150)
plt.show()
print("Plot saved as accuracy_comparison.png")

---
## Summary & Conclusions

| Method | Strengths | Weaknesses |
|--------|-----------|------------|
| **One-Hot Encoding** | Simple, interpretable | Binary only, no frequency, very sparse |
| **Bag of Words** | Captures frequency, fast | No IDF weighting, semantic blindness |
| **TF-IDF** | Down-weights common words, better features | Corpus-dependent, no semantics |

**Key Findings:**
- TF-IDF consistently gives **equal or better accuracy** vs BoW for sentiment classification by suppressing uninformative high-frequency words.
- **Logistic Regression** typically outperforms Naive Bayes on TF-IDF features because it can learn complex decision boundaries.
- The sparse matrices contain **>90% zeros**, highlighting the need for compressed storage (CSR format) or modern dense embeddings (BERT, FastText).
- For production-grade NLP, word embeddings (Word2Vec/BERT) would significantly outperform all three methods here by capturing semantic meaning.